In [2]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

In [ ]:
pip install scikit-optimize # type: ignore

In [ ]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from joblib import dump
from time import time
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# ------------------------
# Paso 2: Cargar los datos
# ------------------------

y = Data_final['isFraud']
x = Data_final.drop(columns=['isFraud']) # anexar base de datos de JESÚS.
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=80,stratify=y)

# ------------------------
# Paso 3: Definir el pipeline
# ------------------------
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# ------------------------
# Paso 4: Definir espacio de búsqueda para BayesSearchCV
# ------------------------
search_spaces = {
    'knn__n_neighbors': Integer(3, 15),
    'knn__weights': Categorical(['uniform', 'distance'])
}

# ------------------------
# Paso 5: Entrenar el modelo con BayesSearchCV
# ------------------------
bayes_knn = BayesSearchCV(
    estimator=pipe_knn,
    search_spaces=search_spaces,
    n_iter=25,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time()
bayes_knn.fit(x_train, y_train)
training_time_knn = time() - start_time

# Guardar el modelo
dump(bayes_knn, 'bayes_knn.joblib')

# ------------------------
# Paso 6: Hacer predicciones
# ------------------------
y_pred_knn = bayes_knn.best_estimator_.predict(x_test)
y_pred_proba_knn = bayes_knn.best_estimator_.predict_proba(x_test)[:, 1]

# ------------------------
# Paso 7: Calcular métricas
# ------------------------
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
accuracy_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
auc_knn = roc_auc_score(y_test, y_pred_proba_knn)

# ------------------------
# Paso 8: Resultados en DataFrame
# ------------------------
resultados_knn = pd.DataFrame({
    'Precision': [f"{precision_knn:.2f}"],
    'Recall': [f"{recall_knn:.2f}"],
    'Accuracy': [f"{accuracy_knn:.2f}"],
    'F1-Score': [f"{f1_knn:.2f}"],
    'AUC': [f"{auc_knn:.2f}"],
    'CPU time (s)': [round(training_time_knn, 2)]
})

# ------------------------
# Paso 9: Mostrar resultados
# ------------------------
print("Métricas para el modelo KNN (Bayesian Optimization):")
display(resultados_knn)

In [11]:
display(resultados_knn)

,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.96,0.97,0.97,0.96,0.78,23844.27


El modelo KNN para detección de fraude muestra resultados sobresalientes, con métricas de rendimiento que sugieren una alta capacidad predictiva. La precisión del 96% indica que las predicciones de transacciones fraudulentas son muy precisas, mientras que el recall del 97% muestra que el modelo captura casi todas las transacciones fraudulentas reales. La accuracy del 97% confirma su excelente desempeño general en la clasificación. El F1-Score de 0.96 equilibra perfectamente precisión y recall. El AUC de 0.78, aunque bueno, sugiere cierta complejidad en la separación perfecta de clases, lo cual es típico en problemas de detección de fraude. El tiempo de CPU extremadamente alto (23,844.27 segundos o ~6.6 horas) es característico de KNN, ya que este algoritmo realiza cálculos de distancia para cada punto de prueba con todos los puntos de entrenamiento, siendo computacionalmente intensivo especialmente con datasets grandes como este de 590,540 registros.